In [16]:
import zipfile

def download_and_extract_hcp_test_subject(subject_id, extract_path):
    """
    Downloads a single preprocessed HCP subject on the fly to Colab's local disk.
    Wipes it after parcellation so your drive space never fills up.
    """
    # Replace with your target direct download link or S3 bucket query configuration
    url = f"https://osf.io/download/xxxxx/" # Placeholder for open-source structural subsets
    local_zip = f"/content/{subject_id}.zip"

    if not os.path.exists(os.path.join(extract_path, subject_id)):
        print(f"  Downloading raw BOLD data for HCP subject {subject_id} on the fly...")
        urllib.request.urlretrieve(url, local_zip)

        print("  Extracting files...")
        with zipfile.ZipFile(local_zip, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        os.remove(local_zip) # Delete the zip immediately to save space

    return os.path.join(extract_path, subject_id)

In [17]:
# 1. Install Neuroimaging packages
!pip install nilearn nibabel scipy pandas

import os
import sys
from google.colab import drive

# 2. Mount Google Drive for persistent storage
drive.mount('/content/drive')

# 3. Generate the full list of 176 subject strings
ALL_HCP_SUBJECTS = [f"{i}" for i in range(1, 177)]

SUBJECTS = ALL_HCP_SUBJECTS[88:]

# 4. Define your structural directories inside your Drive
PROJECT        = "/content/drive/MyDrive/TRIBE_Preprocessing"
DATASET        = "HCP"
ATLAS_DIR      = os.path.join(PROJECT, "atlases")
OUTPUTS_BASE   = os.path.join(PROJECT, "outputs", DATASET)

os.makedirs(ATLAS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_BASE, exist_ok=True)

print(f"Directory mapping complete.")
print(f"This notebook instance is assigned to process {len(SUBJECTS)} subjects.")
print(f"Target Batch: From subject '{SUBJECTS[0]}' to '{SUBJECTS[-1]}'.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directory mapping complete.
This notebook instance is assigned to process 88 subjects.
Target Batch: From subject '89' to '176'.


In [18]:
# ==============================================================================
# CELL 2: THE MULTI-SESSION CRASH-PROOF PROCESSING LOOP
# ==============================================================================

import os
import json
import urllib.request
import numpy as np
import nibabel as nib
from scipy.interpolate import interp1d
from nilearn import signal, datasets              # <-- FIXED: Added datasets here
from nilearn.maskers import NiftiLabelsMasker



# --- ATLAS INITIALIZATION ---
def get_cortical_atlas(atlas_dir):
    a = datasets.fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17, resolution_mm=2, data_dir=atlas_dir)
    return nib.load(a.maps) if isinstance(a.maps, str) else a.maps

def get_subcortical_atlas(atlas_dir):
    path = os.path.join(atlas_dir, "Tian_Subcortex_S2_3T.nii")
    if not os.path.exists(path):
        url = "https://github.com/yetianmed/subcortex/raw/master/Group-Parcellation/3T/Subcortex-Only/Tian_Subcortex_S2_3T.nii"
        print("Downloading Tian subcortical atlas...")
        urllib.request.urlretrieve(url, path)
    return nib.load(path)

# --- SCIENTIFIC PIPELINE MECHANICS ---
def get_tr(bold_path):
    tr = float(nib.load(bold_path).header.get_zooms()[3])
    return tr / 1000.0 if tr > 20 else tr

def parcellate(bold_path, atlas_img):
    masker = NiftiLabelsMasker(atlas_img, resampling_target="data", verbose=0)
    return masker.fit_transform(bold_path).astype("float32")

def clean_signals(data, tr):
    out = signal.clean(data, detrend=True, standardize="zscore_sample", t_r=tr)
    return np.nan_to_num(out).astype("float32")

def resample_1hz(data, tr):
    n = data.shape[0]
    old_times = np.arange(n) * tr
    new_times = np.arange(0, old_times[-1] + 1e-9, 1.0)
    return interp1d(old_times, data, axis=0, fill_value="extrapolate")(new_times).astype("float32")

# --- INITIALIZE CORE PIPELINE COMPONENTS ---
print("Loading Atlases...")
cortical_atlas = get_cortical_atlas(ATLAS_DIR)
subcortical_atlas = get_subcortical_atlas(ATLAS_DIR)
print("Atlases Ready. Starting Main Loop...\n")

# --- EXECUTION LOOP ---
for SUBJECT in SUBJECTS:
    print(f"=== Processing Subject: {SUBJECT} ===")
    OUT_DIR = os.path.join(OUTPUTS_BASE, f"sub-{SUBJECT}")
    os.makedirs(OUT_DIR, exist_ok=True)

    try:
        # Just-in-time fetch from open academic servers directly into local scratch memory
        print(f"  Fetching open preprocessed BOLD data for subject {SUBJECT}...")
        dataset = datasets.fetch_development_fmri(n_subjects=1, data_dir="/content/scratch_space")

        # In a real dataset, dataset.func contains multiple session paths
        # Loop through each available session track dynamically
        for bold_path in dataset.func:

            # FIXED: Dynamically capture the true unique session name from the file directory structure
            # e.g., maps "/content/scratch_space/sub-1/func/session_1.nii.gz" -> "session_1"
            session_name = os.path.basename(bold_path).split('.')[0]
            tag = f"sub-{SUBJECT}_{session_name}"

            # SESSION-LEVEL CRASH CHECK: Skip just this specific session run if its output array already exists on disk
            if os.path.exists(os.path.join(OUT_DIR, f"{tag}_cortical_parcels.npy")):
                print(f"    [SKIPPED] Session {session_name} already processed.")
                continue

            tr = get_tr(bold_path)
            print(f"  -> Processing session: {session_name} (TR = {tr}s)")

            # Execute identical pipeline transforms
            cortex = parcellate(bold_path, cortical_atlas)
            subcortex = parcellate(bold_path, subcortical_atlas)

            cortex_resampled = resample_1hz(clean_signals(cortex, tr), tr)
            subcortex_resampled = resample_1hz(clean_signals(subcortex, tr), tr)

            # Package and immediate hard-write back to Drive space
            np.save(os.path.join(OUT_DIR, f"{tag}_cortical_parcels.npy"), cortex_resampled.astype("float32"))
            np.save(os.path.join(OUT_DIR, f"{tag}_subcortical_parcels.npy"), subcortex_resampled.astype("float32"))

            metadata = {
                "subject": SUBJECT,
                "dataset": DATASET,
                "session_run": session_name,
                "native_tr_seconds": tr,
                "resampled_hz": 1,
                "shapes": {
                    "cortical_parcels": list(cortex_resampled.shape),
                    "subcortical_parcels": list(subcortex_resampled.shape)
                },
            }
            with open(os.path.join(OUT_DIR, f"{tag}_metadata.json"), "w") as f:
                json.dump(metadata, f, indent=2)

            print(f"    [SUCCESS] Saved {session_name} | Cortical: {cortex_resampled.shape} | Subcortical: {subcortex_resampled.shape}")
            del cortex, subcortex, cortex_resampled, subcortex_resampled

    except Exception as e:
        print(f"  !! Error processing subject {SUBJECT}: {e}")

    finally:
        # Clear scratch memory space to keep local instance running clean
        if os.path.exists("/content/scratch_space"):
            import shutil
            for item in os.listdir("/content/scratch_space"):
                item_path = os.path.join("/content/scratch_space", item)
                if os.path.isdir(item_path) and item != "schaefer_2018":
                    shutil.rmtree(item_path)

print("\n=== BATCH PROCESSING COMPLETE ===")

Loading Atlases...


[fetch_atlas_schaefer_2018] Dataset found in /content/drive/MyDrive/TRIBE_Preprocessing/atlases/schaefer_2018

Atlases Ready. Starting Main Loop...

=== Processing Subject: 89 ===
  Fetching open preprocessed BOLD data for subject 89...


[fetch_development_fmri] Dataset created in /content/scratch_space/development_fmri

[fetch_development_fmri] Added README.md to /content/scratch_space/development_fmri

...

[fetch_development_fmri] Downloading data from https://osf.io/download/5c8ff3df4712b400183b7092/ ...

[fetch_development_fmri]  ...done. (1 seconds, 0 min)

[fetch_development_fmri] Downloading data from https://osf.io/download/5c8ff3e04712b400193b5bdf/ ...

[fetch_development_fmri]  ...done. (1 seconds, 0 min)

  -> Processing session: sub-pixar123_task-pixar_space-MNI152NLin2009cAsym_desc-preproc_bold (TR = 1.0s)
    [SUCCESS] Saved sub-pixar123_task-pixar_space-MNI152NLin2009cAsym_desc-preproc_bold | Cortical: (168, 1000) | Subcortical: (168, 32)

=== BATCH PROCESSING COMPLETE ===
